### Topic 4: Different Types of Text Splitters

### What is a Text Splitter?

### Definition

A **Text Splitter** is a component that divides a document into smaller chunks based on a particular chunking strategy.

Unlike a chunking strategy, which describes **how** a document should be split, a text splitter is the actual implementation that performs the splitting.

---

### Why Do We Need Text Splitters?

In the previous topic, we learned different chunking strategies such as Fixed-Size Chunking, Semantic Chunking, and Structure-Aware Chunking.

However, these are only concepts.

When building a RAG application, we need actual code that can split documents according to these strategies.

Instead of writing custom chunking logic every time, libraries such as **LangChain** provide reusable implementations called **Text Splitters**.

These splitters:

- Reduce development effort.
- Handle many edge cases.
- Can easily be swapped without changing the rest of the RAG pipeline.

---

### Commonly Used Text Splitters

The most commonly used text splitters are:

- CharacterTextSplitter
- RecursiveCharacterTextSplitter
- TokenTextSplitter
- MarkdownHeaderTextSplitter

Each splitter follows a different approach for deciding where to split the document.

---

### 1. CharacterTextSplitter

### Definition

CharacterTextSplitter divides a document using a single separator, such as a newline (`\n`) or a blank line (`\n\n`).

If the separator cannot be found before reaching the configured chunk size, it simply cuts the text at the character limit.

---

### Why Do We Need It?

Many documents are already divided into paragraphs.

Instead of cutting text at an exact character count, CharacterTextSplitter first tries to split at natural paragraph boundaries, producing more readable chunks.

---

### How It Works

1. Choose a separator (for example, a blank line).
2. Keep adding text until the chunk reaches the configured size.
3. Split at the separator.
4. If no separator is found within the size limit, perform a hard character cut.

---

### Example

Suppose the document is:

```text
Q: How do I reset my password?

Click "Forgot Password."

Enter your email.

Q: How do I verify my email?

Click the verification link.
```

Using a blank line as the separator produces two chunks.

**Chunk 1**

```text
Q: How do I reset my password?

Click "Forgot Password."

Enter your email.
```

**Chunk 2**

```text
Q: How do I verify my email?

Click the verification link.
```

---

### Advantages

- Simple to configure.
- Fast to execute.
- Works well for documents with consistent paragraph formatting.

---

### Disadvantages

- Relies on a single separator.
- Long paragraphs may still be cut in the middle.
- Character count is only an approximation of token count.

---

### 2. RecursiveCharacterTextSplitter

### Definition

RecursiveCharacterTextSplitter is an improved version of CharacterTextSplitter.

Instead of using only one separator, it tries multiple separators in priority order until it finds the best possible place to split.

---

### Why Do We Need It?

Not every document contains clean paragraph breaks.

Some documents contain only newlines, while others may contain only long sentences or continuous text.

RecursiveCharacterTextSplitter increases the chances of finding a natural split point before resorting to a hard character cut.

---

### How It Works

It tries separators in the following order:

1. Blank lines (`\n\n`)
2. New lines (`\n`)
3. Sentence boundaries
4. Spaces
5. Character limit (last resort)

If one separator cannot produce an appropriate chunk, it automatically tries the next one.

---

### Example

Suppose a document has no blank lines.

CharacterTextSplitter immediately performs a hard character cut.

RecursiveCharacterTextSplitter instead tries:

- New lines
- Sentence boundaries
- Spaces

Only if all of these fail does it cut at the character limit.

As a result, the generated chunks are usually more meaningful.

---

### Advantages

- Produces better chunk boundaries.
- Preserves context more effectively.
- Suitable for most document types.
- Commonly used as the default splitter in many RAG applications.

---

### Disadvantages

- Slightly slower than CharacterTextSplitter.
- Still based on characters rather than tokens.

---

### 3. TokenTextSplitter

### Definition

TokenTextSplitter divides a document according to the number of tokens instead of the number of characters.

It uses the tokenizer associated with the embedding model or language model.

---

### Why Do We Need It?

LLMs process tokens, not characters.

Two pieces of text with the same number of characters may contain very different numbers of tokens.

Character-based chunking can therefore produce chunks that exceed the model's context limit.

TokenTextSplitter guarantees that every chunk stays within the required token limit.

---

### How It Works

1. Convert the document into tokens.
2. Count the tokens.
3. Create chunks containing the configured number of tokens.
4. Convert each token group back into text.

---

### Example

Suppose an embedding model accepts a maximum of **512 tokens**.

Instead of counting characters, TokenTextSplitter creates chunks containing exactly **512 tokens**, ensuring that no chunk exceeds the model's limit.

---

### Advantages

- Respects the model's actual context window.
- Produces accurate chunk sizes.
- Preferred for production systems where token limits are important.

---

### Disadvantages

- Requires tokenization before splitting.
- Slightly more computationally expensive than character-based splitting.

---

### 4. MarkdownHeaderTextSplitter

### Definition

MarkdownHeaderTextSplitter divides a Markdown document according to its header hierarchy instead of using chunk size.

---

### Why Do We Need It?

Markdown documents already contain logical sections identified by headers.

Instead of splitting by size, it preserves these sections, resulting in more meaningful chunks.

---

### How It Works

1. Detect Markdown headers (`#`, `##`, `###`, etc.).
2. Treat each section as a separate chunk.
3. Preserve the header hierarchy as metadata.

---

### Example

Suppose the document is:

```markdown
### Authentication

Reset Password

### OTP

OTP expires after 5 minutes.

### Payments

Refund Policy
```

The splitter creates separate chunks for:

- Authentication
- OTP
- Payments

Each chunk represents a logical section of the document rather than an arbitrary number of characters.

---

### Advantages

- Preserves the document's logical structure.
- Produces highly meaningful chunks.
- Useful for technical documentation, manuals, and knowledge bases.

---

### Disadvantages

- Works only for well-structured Markdown documents.
- Inconsistent or missing headers reduce chunk quality.

---

### Key Takeaways

- A **chunking strategy** describes **how** a document should be divided.
- A **text splitter** is the actual implementation of that strategy.
- Different splitters are suitable for different types of documents.
- Choosing the appropriate splitter depends on the document structure, the embedding model, and the retrieval requirements.

In [1]:
"""
text_splitters.py
--------------------
Hand-written implementations of three named splitter types, built to be
directly comparable to chunker.py's sentence-aware splitter and the four
strategies in chunking_strategies.py.

  character_text_split  -- single separator, hard-cut fallback.
  token_text_split        -- sizes chunks by actual token count, not chars.
  markdown_header_split   -- splits on Markdown headers, attaching the
                              header hierarchy into each chunk's metadata.
"""

import re
from document import make_document


# ----------------------------------------------------------------------
# 1. CharacterTextSplitter -- single separator, hard-cut fallback
# ----------------------------------------------------------------------
def character_text_split(text: str, separator: str = "\n\n", chunk_size: int = 200) -> list:
    pieces = text.split(separator)
    chunks, current = [], ""

    for piece in pieces:
        candidate = (current + separator + piece) if current else piece
        if len(candidate) > chunk_size:
            if current:
                chunks.append(current)
            if len(piece) > chunk_size:
                # fallback: separator didn't help, hard-cut this piece
                for i in range(0, len(piece), chunk_size):
                    chunks.append(piece[i:i + chunk_size])
                current = ""
            else:
                current = piece
        else:
            current = candidate

    if current:
        chunks.append(current)
    return chunks


# ----------------------------------------------------------------------
# 2. TokenTextSplitter -- sizes by token count, not character count.
#    Uses a simple whitespace tokenizer as a stand-in; swap in a real
#    tokenizer (e.g. tiktoken, or the embedding model's own tokenizer)
#    to match this exactly to your actual downstream model.
# ----------------------------------------------------------------------
def _simple_tokenize(text: str) -> list:
    return text.split()  # placeholder tokenizer -- replace with a real one in production


def _simple_detokenize(tokens: list) -> str:
    return " ".join(tokens)


def token_text_split(text: str, chunk_size_tokens: int = 30) -> list:
    tokens = _simple_tokenize(text)
    return [
        _simple_detokenize(tokens[i:i + chunk_size_tokens])
        for i in range(0, len(tokens), chunk_size_tokens)
    ]


# ----------------------------------------------------------------------
# 3. MarkdownHeaderTextSplitter -- splits on # / ## / ### headers,
#    attaching the header hierarchy into each chunk's metadata.
# ----------------------------------------------------------------------
HEADER_RE = re.compile(r"^(#{1,3})\s+(.*)$", re.MULTILINE)


def markdown_header_split(text: str) -> list:
    matches = list(HEADER_RE.finditer(text))
    if not matches:
        return [make_document(text.strip(), {"header_path": []})]

    documents = []
    header_stack = []  # tracks current (level, title) hierarchy

    for i, match in enumerate(matches):
        level = len(match.group(1))
        title = match.group(2).strip()

        # pop any headers at the same or deeper level before pushing this one
        header_stack = [h for h in header_stack if h[0] < level]
        header_stack.append((level, title))

        start = match.end()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(text)
        section_text = text[start:end].strip()

        if section_text:
            documents.append(make_document(
                page_content=section_text,
                metadata={"header_path": [h[1] for h in header_stack]},
            ))

    return documents


if __name__ == "__main__":
    sample = (
        "Premature withdrawal incurs a 1 percent penalty.\n\n"
        "This does not apply if the FD is closed due to the death of the depositor.\n\n"
        "Senior citizens receive an additional 0.5 percent interest on all tenures."
    )

    print("--- CharacterTextSplitter ---")
    for c in character_text_split(sample, separator="\n\n", chunk_size=80):
        print(f"  {c!r}")

    print("\n--- TokenTextSplitter ---")
    for c in token_text_split(sample, chunk_size_tokens=10):
        print(f"  {c!r}")

    markdown_sample = (
        "# FD Policy\n"
        "Overview text about FD policy.\n"
        "## Premature Withdrawal\n"
        "A 1 percent penalty applies.\n"
        "### Death of Depositor Exception\n"
        "No penalty applies in this case.\n"
        "## Senior Citizen Rate\n"
        "An additional 0.5 percent applies.\n"
    )
    print("\n--- MarkdownHeaderTextSplitter ---")
    for d in markdown_header_split(markdown_sample):
        print(f"  header_path={d['metadata']['header_path']}: {d['page_content']!r}")

--- CharacterTextSplitter ---
  'Premature withdrawal incurs a 1 percent penalty.'
  'This does not apply if the FD is closed due to the death of the depositor.'
  'Senior citizens receive an additional 0.5 percent interest on all tenures.'

--- TokenTextSplitter ---
  'Premature withdrawal incurs a 1 percent penalty. This does not'
  'apply if the FD is closed due to the death'
  'of the depositor. Senior citizens receive an additional 0.5 percent'
  'interest on all tenures.'

--- MarkdownHeaderTextSplitter ---
  header_path=['FD Policy']: 'Overview text about FD policy.'
  header_path=['FD Policy', 'Premature Withdrawal']: 'A 1 percent penalty applies.'
  header_path=['FD Policy', 'Premature Withdrawal', 'Death of Depositor Exception']: 'No penalty applies in this case.'
  header_path=['FD Policy', 'Senior Citizen Rate']: 'An additional 0.5 percent applies.'
